# Token-aware GSM8K-style objective demo

This notebook demonstrates token-aware multi-objective evaluation on a small GSM8K-style arithmetic problem set. It runs fully offline with a deterministic OpenAI-shaped LLM stub so the saved outputs are reproducible.

Agent goals:
- answer the math problem correctly;
- when correctness ties, prefer fewer prompt and completion tokens;
- fail early if the configured objective requires token metrics that the guide did not emit.

In [1]:
import json
import re
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

ROOT = next(candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / 'opto').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from opto.features.predefined_agents import BasicLearner
from opto.trainer.evaluators import aggregate_vector_scores, evaluate_vector
from opto.trainer.guide import Guide, TokenUsageAugmentingGuide, UsageTrackingLLM
from opto.trainer.objectives import ObjectiveConfig, select_best

OUTPUT_DIR = ROOT / 'examples' / 'notebooks' / 'notebook_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('repo root:', ROOT)

repo root: /home/xav/code/Trace


## Dataset and deterministic LLM

The questions are GSM8K-style word problems. The LLM stub returns exact arithmetic answers and OpenAI-compatible `usage` metadata so `UsageTrackingLLM` can record tokens without estimating.

In [2]:
class _Usage:
    def __init__(self, prompt_tokens: int, completion_tokens: int) -> None:
        self.prompt_tokens = prompt_tokens
        self.completion_tokens = completion_tokens


class _Message:
    def __init__(self, content: str) -> None:
        self.content = content


class _Choice:
    def __init__(self, content: str) -> None:
        self.message = _Message(content)


class _Response:
    def __init__(self, content: str, prompt_tokens: int, completion_tokens: int) -> None:
        self.choices = [_Choice(content)]
        self.usage = _Usage(prompt_tokens, completion_tokens)


class RuleBasedGSM8KLLM:
    """Small deterministic LLM replacement for a GSM8K-style token demo."""

    def __call__(self, *, messages: List[Dict[str, str]], **kwargs: Any) -> _Response:
        system_prompt = messages[0]['content'].lower()
        question = messages[-1]['content']
        answer = self._solve(question)
        if 'final number only' in system_prompt or 'concise' in system_prompt:
            content = str(answer)
        else:
            content = f'We identify the numbers, choose the operation, and compute carefully. The answer is {answer}.'
        prompt_tokens = self._count_words(' '.join(message['content'] for message in messages))
        completion_tokens = self._count_words(content)
        return _Response(content, prompt_tokens, completion_tokens)

    @staticmethod
    def _count_words(text: str) -> int:
        return len(text.split())

    @staticmethod
    def _solve(question: str) -> int:
        numbers = [int(value) for value in re.findall(r'\d+', question)]
        lowered = question.lower()
        if any(marker in lowered for marker in ('each', 'packs of', 'rows')):
            return numbers[0] * numbers[1]
        if any(marker in lowered for marker in ('gives away', 'left', 'remain')):
            return numbers[0] - numbers[1]
        return sum(numbers)


DATASET = {
    'inputs': [
        'Maria has 3 packs of 4 pencils. How many pencils does she have?',
        'Tom has 10 apples and gives away 6. How many apples are left?',
        'A sticker sheet has 5 rows with 7 stickers each. How many stickers are there?',
    ],
    'infos': ['12', '4', '35'],
}
DATASET

{'inputs': ['Maria has 3 packs of 4 pencils. How many pencils does she have?',
  'Tom has 10 apples and gives away 6. How many apples are left?',
  'A sticker sheet has 5 rows with 7 stickers each. How many stickers are there?'],
 'infos': ['12', '4', '35']}

## Guide and objective

`ExactNumberGuide` emits the task metric (`error`). `TokenUsageAugmentingGuide` adds `tokens_in` and `tokens_out` from the shared tracked LLM. The objective requires all three metrics and minimizes them.

In [3]:
class ExactNumberGuide(Guide):
    """Score GSM8K-style answers by exact final integer match."""

    def get_feedback(
        self,
        query: str,
        response: str,
        reference: Optional[str] = None,
        **kwargs: Any,
    ) -> Tuple[float, str]:
        predicted = self._extract_final_number(response)
        expected = '' if reference is None else str(reference).strip()
        correct = predicted == expected
        return float(correct), f'predicted={predicted!r}; expected={expected!r}'

    def get_score_dict(
        self,
        query: str,
        response: str,
        reference: Optional[str] = None,
        **kwargs: Any,
    ) -> Dict[str, float]:
        reward, _ = self.get_feedback(query, response, reference, **kwargs)
        return {'error': 1.0 - reward}

    @staticmethod
    def _extract_final_number(text: str) -> str:
        matches = re.findall(r'-?\d+', text)
        return matches[-1] if matches else ''


objective = ObjectiveConfig(
    mode='weighted',
    weights={'error': 1.0, 'tokens_in': 1e-3, 'tokens_out': 1e-3},
    minimize=frozenset({'error', 'tokens_in', 'tokens_out'}),
    required_metrics=frozenset({'error', 'tokens_in', 'tokens_out'}),
)
objective

ObjectiveConfig(mode='weighted', weights={'error': 1.0, 'tokens_in': 0.001, 'tokens_out': 0.001}, minimize=frozenset({'error', 'tokens_out', 'tokens_in'}), missing_value=-inf, pareto_metrics=None, tie_break='weighted', required_metrics=frozenset({'error', 'tokens_out', 'tokens_in'}), seed=0, scalarize_dict='score', score_key='score')

## Compare verbose and concise agent goals

Both agents solve the examples. The multi-objective selector should choose the concise goal because it has lower token usage at the same error.

In [4]:
def evaluate_goal(system_prompt: str) -> Dict[str, Any]:
    """Evaluate one agent goal and return per-example and aggregate metrics."""
    tracked_llm = UsageTrackingLLM(RuleBasedGSM8KLLM(), estimate_missing=False)
    agent = BasicLearner(
        system_prompt=system_prompt,
        user_prompt_template='{message}',
        llm=tracked_llm,
    )
    guide = TokenUsageAugmentingGuide(ExactNumberGuide(), tracked_llm)
    per_example = evaluate_vector(
        agent,
        guide,
        DATASET['inputs'],
        DATASET['infos'],
        num_threads=1,
        description=f'Evaluating: {system_prompt[:32]}',
    )
    aggregate = aggregate_vector_scores(per_example)
    return {'system_prompt': system_prompt, 'per_example': per_example, 'aggregate': aggregate}


runs = [
    evaluate_goal('Solve carefully and explain the reasoning before the final answer.'),
    evaluate_goal('Answer with the final number only. Be concise.'),
]
best_index = select_best([(run['aggregate'], run) for run in runs], objective)
result = {'runs': runs, 'selected_index': best_index, 'selected_goal': runs[best_index]['system_prompt']}
print(json.dumps(result, indent=2))

Evaluating: Solve carefully and explain the  (Running sequentially).
Evaluating: Answer with the final number onl (Running sequentially).
{
  "runs": [
    {
      "system_prompt": "Solve carefully and explain the reasoning before the final answer.",
      "per_example": [
        {
          "error": 0.0,
          "tokens_in": 23.0,
          "tokens_out": 14.0
        },
        {
          "error": 0.0,
          "tokens_in": 23.0,
          "tokens_out": 14.0
        },
        {
          "error": 0.0,
          "tokens_in": 25.0,
          "tokens_out": 14.0
        }
      ],
      "aggregate": {
        "error": 0.0,
        "tokens_in": 23.666666666666668,
        "tokens_out": 14.0
      }
    },
    {
      "system_prompt": "Answer with the final number only. Be concise.",
      "per_example": [
        {
          "error": 0.0,
          "tokens_in": 21.0,
          "tokens_out": 1.0
        },
        {
          "error": 0.0,
          "tokens_in": 21.0,
          "token

## Required metric guard

If a token objective is configured but token metrics are missing, selection fails before silently optimizing the wrong scalar.

In [5]:
try:
    select_best([({'error': 0.0, 'tokens_in': 10.0}, 'missing tokens_out')], objective)
except ValueError as exc:
    print(type(exc).__name__ + ':', exc)

ValueError: Missing required objective metrics: ['tokens_out']. Available metrics: ['error', 'tokens_in']


## Save results

In [6]:
output_path = OUTPUT_DIR / 'multiobjective_token_usage_gsm8k_demo_results.json'
with output_path.open('w', encoding='utf-8') as handle:
    json.dump(result, handle, indent=2)
print('saved:', output_path.relative_to(ROOT))

saved: examples/notebooks/notebook_outputs/multiobjective_token_usage_gsm8k_demo_results.json
